In [37]:
!pip install plotly pandas reportlab

In [38]:
import pandas as pd

# Using the direct link to the raw CSV file
df = pd.read_csv('https://gist.githubusercontent.com/SharmaAshwini/8bd642f6c46792a9c40c8ccad60391e9/raw/Sample_Superstore.csv', encoding='latin1')

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,08/11/16,11/11/16,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,08/11/16,11/11/16,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12/06/16,16/06/16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11/10/15,18/10/15,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11/10/15,18/10/15,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [39]:
# Convert date column
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Create Year-Month for trend
df['Year-Month'] = df['Order Date'].dt.to_period('M').astype(str)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   object        
 2   Order Date     9994 non-null   datetime64[ns]
 3   Ship Date      9994 non-null   object        
 4   Ship Mode      9994 non-null   object        
 5   Customer ID    9994 non-null   object        
 6   Customer Name  9994 non-null   object        
 7   Segment        9994 non-null   object        
 8   Country        9994 non-null   object        
 9   City           9994 non-null   object        
 10  State          9994 non-null   object        
 11  Postal Code    9994 non-null   int64         
 12  Region         9994 non-null   object        
 13  Product ID     9994 non-null   object        
 14  Category       9994 non-null   object        
 15  Sub-Category   9994 n

/tmp/ipykernel_1210/3227352801.py:2: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [40]:
total_revenue = df['Sales'].sum()
top_product = df.groupby('Product Name')['Sales'].sum().idxmax()
top_region = df.groupby('Region')['Sales'].sum().idxmax()

print("Total Revenue:", total_revenue)
print("Top Product:", top_product)
print("Top Region:", top_region)

Total Revenue: 2297200.8603000003
Top Product: Canon imageCLASS 2200 Advanced Copier
Top Region: West


In [41]:
import plotly.express as px

# Revenue by Region
fig_region = px.bar(df.groupby('Region')['Sales'].sum().reset_index(),
                    x='Region', y='Sales',
                    title='Sales by Region')

fig_region.show()
top_products = df.groupby('Product Name')['Sales'].sum().nlargest(10).reset_index()

fig_products = px.bar(top_products,
                      x='Sales', y='Product Name',
                      orientation='h',
                      title='Top 10 Products')

fig_products.show()
# Sales Trend
trend = df.groupby('Year-Month')['Sales'].sum().reset_index()

fig_trend = px.line(trend,
                    x='Year-Month', y='Sales',
                    title='Sales Trend Over Time')

fig_trend.show()

In [42]:
import ipywidgets as widgets#Used to create dropdown filters
from IPython.display import display

# Dropdowns
region_filter = widgets.Dropdown(options=df['Region'].unique(), description='Region:')
category_filter = widgets.Dropdown(options=df['Category'].unique(), description='Category:')

def update_dashboard(region, category):
    filtered = df[(df['Region'] == region) & (df['Category'] == category)]

    fig = px.bar(filtered.groupby('Sub-Category')['Sales'].sum().reset_index(),
                 x='Sub-Category', y='Sales',
                 title=f'Sales in {region} - {category}')

    fig.show()

widgets.interact(update_dashboard,
                 region=region_filter,
                 category=category_filter)

interactive(children=(Dropdown(description='Region:', options=('South', 'West', 'Central', 'East'), value='Sou…

<function __main__.update_dashboard(region, category)>

In [47]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph

doc = SimpleDocTemplate("/content/Sales_Report.pdf") # Changed path to local /content/

content = []

content.append(Paragraph(f"Total Revenue: {total_revenue}", None))
content.append(Paragraph(f"Top Product: {top_product}", None))
content.append(Paragraph(f"Top Region: {top_region}", None))

doc.build(content)

print("PDF Generated and saved to local Colab environment: Sales_Report.pdf")

PDF Generated and saved to local Colab environment: Sales_Report.pdf
